In [3]:
import os
import pandas as pd

# -------------------------
# BASE_DIR 자동 결정
# -------------------------
# 1) 현재 폴더에 result_1이 있으면 -> 지금이 5_baseline_prediction 폴더라고 판단
# 2) 아니면 ./5_baseline_prediction 아래를 사용
if os.path.isdir("./result_1"):
    BASE_DIR = "."
elif os.path.isdir("./5_baseline_prediction/result_1"):
    BASE_DIR = "./5_baseline_prediction"
else:
    raise FileNotFoundError(
        "Cannot find result_1 folder. "
        "Run this notebook in the parent folder or inside 5_baseline_prediction."
    )

RESULT_DIRS = [f"result_{i}" for i in range(1, 6)]

BASE_FILES = [
    "results_O.csv",          # 알파벳 O
    "results_F.csv",
    "results_OF.csv",
    "results_OFR.csv",
    "results_OFR_trials.csv",
]

OUT_DIR = os.path.join(BASE_DIR, "avg_results")
os.makedirs(OUT_DIR, exist_ok=True)


def file_for_result(base_file: str, idx: int) -> str:
    if idx == 1:
        return base_file
    stem, ext = os.path.splitext(base_file)
    return f"{stem}_{idx}{ext}"


def average_one_kind(base_file: str):
    dfs = []
    for idx, rdir in enumerate(RESULT_DIRS, start=1):
        fname = file_for_result(base_file, idx)
        path = os.path.join(BASE_DIR, rdir, fname)

        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing file: {path}")

        dfs.append(pd.read_csv(path))

    # 컬럼 동일성 체크
    cols0 = dfs[0].columns.tolist()
    for idx, df in enumerate(dfs, start=1):
        if df.columns.tolist() != cols0:
            raise ValueError(f"Column mismatch: {base_file} at result_{idx}")

    numeric_cols = dfs[0].select_dtypes(include="number").columns.tolist()
    non_numeric_cols = [c for c in cols0 if c not in numeric_cols]

    concat_df = pd.concat(dfs, ignore_index=True)

    if non_numeric_cols:
        avg_df = (
            concat_df
            .groupby(non_numeric_cols, dropna=False, as_index=False)[numeric_cols]
            .mean()
        )
    else:
        avg_df = concat_df[numeric_cols].mean().to_frame().T

    out_path = os.path.join(OUT_DIR, base_file.replace(".csv", "_AVG.csv"))
    avg_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")


if __name__ == "__main__":
    print(f"[INFO] BASE_DIR = {BASE_DIR}")
    for bf in BASE_FILES:
        print(f"Processing: {bf}")
        average_one_kind(bf)
    print("Done.")


[INFO] BASE_DIR = .
Processing: results_O.csv
[SAVED] .\avg_results\results_O_AVG.csv
Processing: results_F.csv
[SAVED] .\avg_results\results_F_AVG.csv
Processing: results_OF.csv
[SAVED] .\avg_results\results_OF_AVG.csv
Processing: results_OFR.csv
[SAVED] .\avg_results\results_OFR_AVG.csv
Processing: results_OFR_trials.csv
[SAVED] .\avg_results\results_OFR_trials_AVG.csv
Done.
